## 1. Load the dataset

In [16]:
from pathlib import Path
import plotly.express as px
import pandas as pd

In [ ]:
cwd = Path.cwd()
root = cwd if (cwd / "data").exists() else cwd.parent

candidates = [
    root / "data" / "mind@work" / "mental_health_dataset" / "Impact_of_Remote_Work_on_Mental_Health.csv"
]

data_file = next((p for p in candidates if p.exists()), None)
assert data_file is not None, f"Dataset not found. Checked:\n- " + "\n- ".join(map(str, candidates))
data_file

PosixPath('/Users/huongle/Documents/GitHub/Mind_At_Work_Website/data/mind@work/mental_health_dataset/Impact_of_Remote_Work_on_Mental_Health.csv')

In [18]:
df = pd.read_csv(data_file)
print(f"rows: {len(df):,} | cols: {df.shape[1]}")
df.head()

rows: 5,000 | cols: 20


,Employee_ID,Age,Gender,Job_Role,Industry,Years_of_Experience,Work_Location,Hours_Worked_Per_Week,Number_of_Virtual_Meetings,Work_Life_Balance_Rating,Stress_Level,Mental_Health_Condition,Access_to_Mental_Health_Resources,Productivity_Change,Social_Isolation_Rating,Satisfaction_with_Remote_Work,Company_Support_for_Remote_Work,Physical_Activity,Sleep_Quality,Region
0,EMP0001,32,Non-binary,HR,Healthcare,13,Hybrid,47,7,2,Medium,Depression,No,Decrease,1,Unsatisfied,1,Weekly,Good,Europe
1,EMP0002,40,Female,Data Scientist,IT,3,Remote,52,4,1,Medium,Anxiety,No,Increase,3,Satisfied,2,Weekly,Good,Asia
2,EMP0003,59,Non-binary,Software Engineer,Education,22,Hybrid,46,11,5,Medium,Anxiety,No,No Change,4,Unsatisfied,5,NaN,Poor,North America
3,EMP0004,27,Male,Software Engineer,Finance,20,Onsite,32,8,4,High,Depression,Yes,Increase,3,Unsatisfied,3,NaN,Poor,Europe
4,EMP0005,49,Male,Sales,Consulting,32,Onsite,35,12,2,High,NaN,Yes,Decrease,3,Unsatisfied,3,Weekly,Average,North America


In [19]:
print("Shape of dataset:", df.shape)
df.head()

Shape of dataset: (5000, 20)


,Employee_ID,Age,Gender,Job_Role,Industry,Years_of_Experience,Work_Location,Hours_Worked_Per_Week,Number_of_Virtual_Meetings,Work_Life_Balance_Rating,Stress_Level,Mental_Health_Condition,Access_to_Mental_Health_Resources,Productivity_Change,Social_Isolation_Rating,Satisfaction_with_Remote_Work,Company_Support_for_Remote_Work,Physical_Activity,Sleep_Quality,Region
0,EMP0001,32,Non-binary,HR,Healthcare,13,Hybrid,47,7,2,Medium,Depression,No,Decrease,1,Unsatisfied,1,Weekly,Good,Europe
1,EMP0002,40,Female,Data Scientist,IT,3,Remote,52,4,1,Medium,Anxiety,No,Increase,3,Satisfied,2,Weekly,Good,Asia
2,EMP0003,59,Non-binary,Software Engineer,Education,22,Hybrid,46,11,5,Medium,Anxiety,No,No Change,4,Unsatisfied,5,NaN,Poor,North America
3,EMP0004,27,Male,Software Engineer,Finance,20,Onsite,32,8,4,High,Depression,Yes,Increase,3,Unsatisfied,3,NaN,Poor,Europe
4,EMP0005,49,Male,Sales,Consulting,32,Onsite,35,12,2,High,NaN,Yes,Decrease,3,Unsatisfied,3,Weekly,Average,North America


In [20]:
df["Physical_Activity"] = df["Physical_Activity"].fillna("No Activity")
df["Mental_Health_Condition"] = df["Mental_Health_Condition"].fillna("No Condition")
# Confirm fixes
print(df.isna().sum())

Employee_ID                          0
Age                                  0
Gender                               0
Job_Role                             0
Industry                             0
Years_of_Experience                  0
Work_Location                        0
Hours_Worked_Per_Week                0
Number_of_Virtual_Meetings           0
Work_Life_Balance_Rating             0
Stress_Level                         0
Mental_Health_Condition              0
Access_to_Mental_Health_Resources    0
Productivity_Change                  0
Social_Isolation_Rating              0
Satisfaction_with_Remote_Work        0
Company_Support_for_Remote_Work      0
Physical_Activity                    0
Sleep_Quality                        0
Region                               0
dtype: int64


## 2. Check column info

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 20 columns):
 #   Column                             Non-Null Count  Dtype 
---  ------                             --------------  ----- 
 0   Employee_ID                        5000 non-null   object
 1   Age                                5000 non-null   int64 
 2   Gender                             5000 non-null   object
 3   Job_Role                           5000 non-null   object
 4   Industry                           5000 non-null   object
 5   Years_of_Experience                5000 non-null   int64 
 6   Work_Location                      5000 non-null   object
 7   Hours_Worked_Per_Week              5000 non-null   int64 
 8   Number_of_Virtual_Meetings         5000 non-null   int64 
 9   Work_Life_Balance_Rating           5000 non-null   int64 
 10  Stress_Level                       5000 non-null   object
 11  Mental_Health_Condition            5000 non-null   object
 12  Access

what we can see here:

Rows: 5000

Columns: 20

Numeric columns (int64): 7 → Age, Years_of_Experience, Hours_Worked_Per_Week, Number_of_Virtual_Meetings, Work_Life_Balance_Rating, Social_Isolation_Rating, Company_Support_for_Remote_Work

Categorical columns (object): 13 → Gender, Job_Role, Industry, Stress_Level, etc.

Missing values:
- All other columns → 5000 non-null (no missing)

In [22]:
df["Mental_Health_Condition"].unique()[:20]   # peek at first 20 unique values
df["Mental_Health_Condition"].isna().sum()    # how many pandas sees as NaN
df["Mental_Health_Condition"].value_counts(dropna=False)  # include NaN


Mental_Health_Condition
Burnout         1280
Anxiety         1278
Depression      1246
No Condition    1196
Name: count, dtype: int64

## 3. Change datatypes

Work_Life_Balance_Rating, Social_Isolation_Rating, Company_Support_for_Remote_Work these are ordinal/categorical ratings stored as numbers.

Convert rating variables (currently int64) to categorical, object to categorical

In [23]:
rating_cols = [
    "Work_Life_Balance_Rating",
    "Social_Isolation_Rating",
    "Company_Support_for_Remote_Work"
]

for col in rating_cols:
    df[col] = df[col].astype("category")

In [24]:
cat_cols = df.select_dtypes(include="object").columns
for col in cat_cols:
    df[col] = df[col].astype("category")

In [25]:
df.dtypes

Employee_ID                          category
Age                                     int64
Gender                               category
Job_Role                             category
Industry                             category
Years_of_Experience                     int64
Work_Location                        category
Hours_Worked_Per_Week                   int64
Number_of_Virtual_Meetings              int64
Work_Life_Balance_Rating             category
Stress_Level                         category
Mental_Health_Condition              category
Access_to_Mental_Health_Resources    category
Productivity_Change                  category
Social_Isolation_Rating              category
Satisfaction_with_Remote_Work        category
Company_Support_for_Remote_Work      category
Physical_Activity                    category
Sleep_Quality                        category
Region                               category
dtype: object

## 4. Missing value management


No missing value

## 5. Checking duplicate: No any duplicate

In [26]:
df.duplicated().sum()

0

## 6. Look for strange/outlier values: Sanity check

In [27]:
df.describe()

,Age,Years_of_Experience,Hours_Worked_Per_Week,Number_of_Virtual_Meetings
count,5000.000000,5000.000000,5000.000000,5000.000000
mean,40.995000,17.810200,39.614600,7.559000
std,11.296021,10.020412,11.860194,4.636121
min,22.000000,1.000000,20.000000,0.000000
25%,31.000000,9.000000,29.000000,4.000000
50%,41.000000,18.000000,40.000000,8.000000
75%,51.000000,26.000000,50.000000,12.000000
max,60.000000,35.000000,60.000000,15.000000


This step is basically a sanity check → making sure the data doesn’t have impossible values before analysis.

Example here:
- Age: 22–60 => reasonable for workers.
- Years_of_Experience: 1–35 => realistic.
- Hours_Worked_Per_Week: 20–60 => possible but >60 would be suspicious.
- Number_of_Virtual_Meetings: 0–15 => makes sense.

## 7. Categorical sanity check

In [28]:
for col in df.select_dtypes(include=["object", "category"]).columns:
    print(col, df[col].unique()[:10])

Employee_ID ['EMP0001', 'EMP0002', 'EMP0003', 'EMP0004', 'EMP0005', 'EMP0006', 'EMP0007', 'EMP0008', 'EMP0009', 'EMP0010']
Categories (5000, object): ['EMP0001', 'EMP0002', 'EMP0003', 'EMP0004', ..., 'EMP4997', 'EMP4998', 'EMP4999', 'EMP5000']
Gender ['Non-binary', 'Female', 'Male', 'Prefer not to say']
Categories (4, object): ['Female', 'Male', 'Non-binary', 'Prefer not to say']
Job_Role ['HR', 'Data Scientist', 'Software Engineer', 'Sales', 'Marketing', 'Designer', 'Project Manager']
Categories (7, object): ['Data Scientist', 'Designer', 'HR', 'Marketing', 'Project Manager', 'Sales', 'Software Engineer']
Industry ['Healthcare', 'IT', 'Education', 'Finance', 'Consulting', 'Manufacturing', 'Retail']
Categories (7, object): ['Consulting', 'Education', 'Finance', 'Healthcare', 'IT', 'Manufacturing', 'Retail']
Work_Location ['Hybrid', 'Remote', 'Onsite']
Categories (3, object): ['Hybrid', 'Onsite', 'Remote']
Work_Life_Balance_Rating [2, 1, 5, 4, 3]
Categories (5, int64): [1, 2, 3, 4, 5]
S

This check ensures:
- No duplicates or spelling errors (e.g., "Remote" vs "remote").
- Categories are logical and consistent.
- Know exactly how many unique options each variable has

In [29]:
for col in df.select_dtypes(include=["category"]).columns:
    print(f"\nValue counts for {col}:")
    print(df[col].value_counts())


Value counts for Employee_ID:
Employee_ID
EMP0001    1
EMP3331    1
EMP3338    1
EMP3337    1
EMP3336    1
          ..
EMP1667    1
EMP1666    1
EMP1665    1
EMP1664    1
EMP5000    1
Name: count, Length: 5000, dtype: int64

Value counts for Gender:
Gender
Female               1274
Male                 1270
Prefer not to say    1242
Non-binary           1214
Name: count, dtype: int64

Value counts for Job_Role:
Job_Role
Project Manager      738
Sales                733
Designer             723
HR                   716
Software Engineer    711
Data Scientist       696
Marketing            683
Name: count, dtype: int64

Value counts for Industry:
Industry
Finance          747
IT               746
Healthcare       728
Retail           726
Education        690
Manufacturing    683
Consulting       680
Name: count, dtype: int64

Value counts for Work_Location:
Work_Location
Remote    1714
Hybrid    1649
Onsite    1637
Name: count, dtype: int64

Value counts for Work_Life_Balance_Rating:
W

## 8. Drop column

Drop employee ID since we don't use it

In [30]:
df = df.drop(columns=["Employee_ID"])
print(df.shape)
print(df.columns)

(5000, 19)
Index(['Age', 'Gender', 'Job_Role', 'Industry', 'Years_of_Experience',
       'Work_Location', 'Hours_Worked_Per_Week', 'Number_of_Virtual_Meetings',
       'Work_Life_Balance_Rating', 'Stress_Level', 'Mental_Health_Condition',
       'Access_to_Mental_Health_Resources', 'Productivity_Change',
       'Social_Isolation_Rating', 'Satisfaction_with_Remote_Work',
       'Company_Support_for_Remote_Work', 'Physical_Activity', 'Sleep_Quality',
       'Region'],
      dtype='object')


## 9. Create file with processing data


In [31]:
# Save the cleaned (but not encoded) dataset
output_path = "/Users/huongle/Documents/GitHub/Mind-At-Work/data/Process data/Cleaned_remote_work.csv"
df.to_csv(output_path, index=False)

print(f"Cleaned dataset saved to: {output_path}")

Cleaned dataset saved to: /Users/huongle/Documents/GitHub/Mind-At-Work/data/Process data/Cleaned_remote_work.csv
